# Kandinsky V2A — AWS Meeting Readiness

## Purpose

This notebook reads the ZIP from the **already completed Colab Persistent Worker Pilot**. It does not load Kandinsky, generate audio, launch AWS resources, or need AWS credentials. It summarizes observed load time, request times, GPU memory, and host memory so we can discuss a bounded AWS pilot with evidence.

Run it in Colab, point it to the downloaded `kandinsky_v2a_persistent_worker_*.zip`, and review the generated meeting brief. If that ZIP is already in `/content`, the upload dialog is skipped.

The first AWS candidate is one `p5.4xlarge` (one H100, 80 GiB GPU memory, 256 GiB host memory), subject to region-specific availability, quota, and price confirmation. This is a candidate, not a purchase decision. [AWS EC2 specifications](https://docs.aws.amazon.com/ec2/latest/instancetypes/ac.html).

Important: Colab measurements help size the first AWS test, but do not predict AWS latency exactly. We must repeat the same workload on AWS.

## Step 1 — Select the previous pilot report

The ZIP is produced by `Kandinsky_V2A_AWS_Persistent_Worker_Pilot.ipynb`. This cell reuses it if present in the current Colab runtime. Otherwise, upload that report ZIP only; no video or model files are needed.

In [ ]:
from pathlib import Path
import csv, json, statistics, zipfile

REPORT_ZIP = None  # Optional: set to Path('/content/your_report.zip') if you know its path.
if REPORT_ZIP is not None:
    REPORT_ZIP = Path(REPORT_ZIP)
else:
    candidates = sorted(Path('/content').glob('kandinsky_v2a_persistent_worker_*.zip'), key=lambda p: p.stat().st_mtime)
    if candidates:
        REPORT_ZIP = candidates[-1]
        print('Using existing report:', REPORT_ZIP)
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Upload exactly one Persistent Worker Pilot report ZIP.')
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        REPORT_ZIP = Path('/content') / Path(uploaded_name).name
        REPORT_ZIP.write_bytes(uploaded_bytes)

if not REPORT_ZIP.is_file():
    raise FileNotFoundError(f'Report ZIP not found: {REPORT_ZIP}')
print('Report ZIP:', REPORT_ZIP, '| size:', round(REPORT_ZIP.stat().st_size / 1024**2, 2), 'MiB')


## Step 2 — Extract the metrics

We keep GPU memory and host RAM separate. `nvidia-smi` reports whole-card use; PyTorch reports the worker's allocated and reserved peaks. Host memory is system-wide used RAM. The earlier pilot ZIP does not store total host RAM, so this notebook reports its measured peak but does not infer a percentage.

In [ ]:
with zipfile.ZipFile(REPORT_ZIP) as archive:
    names = set(archive.namelist())
    required = {'manifest.json', 'worker_report.json', 'resources/worker_resources.csv'}
    missing = required - names
    if missing:
        raise RuntimeError(f'This is not the expected Persistent Worker Pilot ZIP. Missing: {sorted(missing)}')
    manifest = json.loads(archive.read('manifest.json'))
    worker = json.loads(archive.read('worker_report.json'))
    resource_rows = list(csv.DictReader(archive.read('resources/worker_resources.csv').decode('utf-8').splitlines()))

def nums(rows, key):
    out = []
    for row in rows:
        try:
            if row.get(key) not in (None, ''):
                out.append(float(row[key]))
        except (TypeError, ValueError):
            pass
    return out

requests = worker.get('requests', [])
request_seconds = [float(r['end_to_end_seconds']) for r in requests if r.get('end_to_end_seconds') is not None]
warm_seconds = request_seconds[1:]  # Request 1 includes any first-call overhead after the model is loaded.
gpu_sample_peak_mib = max(nums(resource_rows, 'gpu_used_mib'), default=None)
gpu_total_mib = max(nums(resource_rows, 'gpu_total_mib'), default=None)
gpu_alloc_peak_mib = max([float(r['torch_peak_allocated_mib']) for r in requests if r.get('torch_peak_allocated_mib') is not None], default=None)
gpu_reserved_peak_mib = max([float(r['torch_peak_reserved_mib']) for r in requests if r.get('torch_peak_reserved_mib') is not None], default=None)
host_peak_mib = max(nums(resource_rows, 'host_memory_used_mib'), default=None)
gpu_util_peak = max(nums(resource_rows, 'gpu_utilization_percent'), default=None)
wall_seconds = max(nums(resource_rows, 'elapsed_seconds'), default=None)

def gib(value):
    return None if value is None else value / 1024

metrics = {
    'source_gpu': manifest.get('gpu_name', worker.get('gpu', 'unknown')),
    'source_gpu_total_gib': gib(gpu_total_mib),
    'checkpoint': worker.get('checkpoint', 'not recorded'),
    'model_load_seconds': worker.get('load_seconds', manifest.get('model_load_seconds')),
    'request_seconds': request_seconds,
    'warm_request_median_seconds': statistics.median(warm_seconds) if warm_seconds else None,
    'completed_jobs': len(requests),
    'sampled_gpu_used_peak_gib': gib(gpu_sample_peak_mib),
    'torch_allocated_peak_gib': gib(gpu_alloc_peak_mib),
    'torch_reserved_peak_gib': gib(gpu_reserved_peak_mib),
    'host_system_used_peak_gib': gib(host_peak_mib),
    'peak_gpu_utilization_percent': gpu_util_peak,
    'worker_wall_seconds': wall_seconds,
    'source_commit': worker.get('source_commit'),
    'checkpoint_revisions': worker.get('checkpoint_revisions', []),
    'input_video_sha256': worker.get('input_video_sha256'),
    'error': worker.get('error'),
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))


## Step 3 — Enter the AWS quote when available

Do not guess the hourly price here. Ask AWS for the selected region and enter the quoted On-Demand rate. The estimate below covers EC2 compute only; disk, S3, data transfer, taxes, and any support charges are separate. AWS Budgets alerts are not real-time shutdown controls; pair alerts with an explicit stop/terminate procedure.

In [ ]:
AWS_REGION = 'to_be_confirmed'      # Fill in during/after the AWS meeting.
P5_4XLARGE_HOURLY_USD = None         # Enter the quoted regional rate; leave None until confirmed.
PILOT_HOURS = 3.0                   # Proposed maximum first-session window, not a commitment.

if P5_4XLARGE_HOURLY_USD is None:
    print(f'Compute quote pending for {AWS_REGION}. Ask AWS for p5.4xlarge On-Demand price.')
else:
    print(f'EC2 compute-only estimate: ${P5_4XLARGE_HOURLY_USD * PILOT_HOURS:.2f} '
          f'({PILOT_HOURS:.1f} h × ${P5_4XLARGE_HOURLY_USD:.4f}/h) in {AWS_REGION}.')
    print('Add storage, transfer, and applicable taxes separately.')


## Step 4 — Generate the AWS meeting brief

This brief separates what we have already observed from what AWS still needs to validate. It does not claim Colab performance equals AWS performance.

In [ ]:
def fmt(value, unit='', digits=2):
    return 'not recorded' if value is None else f'{value:.{digits}f}{unit}'

load_s = metrics['model_load_seconds']
warm_s = metrics['warm_request_median_seconds']
gpu_peak = metrics['sampled_gpu_used_peak_gib']
host_peak = metrics['host_system_used_peak_gib']
price_line = ('Not quoted yet' if P5_4XLARGE_HOURLY_USD is None else
              f'${P5_4XLARGE_HOURLY_USD:.4f}/h in {AWS_REGION}; compute-only pilot estimate ${P5_4XLARGE_HOURLY_USD * PILOT_HOURS:.2f}.')

brief = f'''# AWS Meeting Brief — Kandinsky Video-to-Audio Pilot

## Current evidence from the Colab persistent-worker run
- GPU used for this run: {metrics['source_gpu']} ({fmt(metrics['source_gpu_total_gib'], ' GiB total')}).
- Model load time: {fmt(load_s, ' s')}.
- Completed sequential jobs: {metrics['completed_jobs']}.
- Request times: {', '.join(f'{x:.1f}s' for x in request_seconds) if request_seconds else 'not recorded'}.
- Warm request median: {fmt(warm_s, ' s')}.
- Sampled whole-GPU memory peak: {fmt(gpu_peak, ' GiB')}.
- PyTorch allocated / reserved peaks: {fmt(metrics['torch_allocated_peak_gib'], ' GiB')} / {fmt(metrics['torch_reserved_peak_gib'], ' GiB')}.
- Sampled host system RAM used peak: {fmt(host_peak, ' GiB')} (total host RAM was not included in that ZIP).
- Worker wall time: {fmt(metrics['worker_wall_seconds'], ' s')}.
- Input SHA-256: {metrics['input_video_sha256'] or 'not recorded'}.

## Why AWS?
1. Keep one GPU worker and the model loaded between requests instead of depending on an interactive Colab session.
2. Measure cold model-load time, warm request latency, peak RAM/VRAM, stability, and cost per completed job under a controlled deployment environment.
3. Prepare a path from a one-machine pilot to a private API/queue later, if measured demand justifies it.

AWS does not inherently improve the generated audio. Audio quality and synchronization remain model/configuration questions; AWS validates operational reliability, capacity, latency, and cost.

## What we propose to AWS
- One `p5.4xlarge` On-Demand instance for a bounded first pilot; AWS lists one H100 with 80 GiB GPU memory and 256 GiB host memory. Confirm region, quota, and capacity before launch.
- Maximum initial window: {PILOT_HOURS:.1f} hours, then stop/terminate and verify that it is no longer running.
- Keep the same Kandinsky source commit, checkpoint revision, input video, prompt, and seeds for comparable results.
- Run a small smoke test, then 5–10 sequential requests through one persistent worker; observe whether memory stabilizes and collect p50/p95 latency after the cold first request.
- Use private S3 for inputs/outputs and Systems Manager Session Manager for access without public inbound SSH; add least-privilege IAM and CloudWatch logs/metrics.
- No training, multi-GPU cluster, public endpoint, or autoscaling in this first pilot.

## Cost worksheet
- Region: {AWS_REGION}
- `p5.4xlarge` On-Demand quote: {price_line}
- Other cost lines to confirm: EBS volume, S3 storage/requests, data transfer, tax, and support.
- Guardrail: time-box the instance and explicitly stop/terminate it; budget alerts can arrive after billing data refresh.

## Questions for AWS
1. Is one `p5.4xlarge` available in our preferred region, and what quota request is required?
2. What is the exact regional On-Demand price and a 3-hour compute-only estimate? What additional storage/network charges should we include?
3. Which supported Linux image/driver path is suitable for our pinned Kandinsky Python/CUDA environment?
4. Can the instance stay in a private subnet and be managed with Session Manager, with S3 access restricted to one bucket/prefix?
5. What automated stop/termination and cost-alert controls do they recommend for a short pilot?
6. Are AWS credits or architecture-support hours available for this one-GPU proof-of-operation?

## Success criteria for the AWS pilot
- Model loads once, then 5–10 sequential 5-second jobs complete without OOM or worker restart.
- Record cold-load time, first-job time, warm p50/p95, peak GPU/host memory, and exact software/checkpoint revisions.
- Listening review confirms the same scene match and synchronization are preserved; compare sound quality separately.
- Compute cost per completed job is calculated from actual elapsed instance time, not a guessed model token cost.

## Limits of this evidence
- This ZIP reflects one Colab environment and three jobs; it is a sizing hint, not an AWS benchmark or a production concurrency test.
- Whole-GPU `nvidia-smi` use includes runtime/context overhead. PyTorch allocated/reserved values are model-process measurements.
- A 3-hour pilot window is a planning cap; the actual duration and cost depend on region, image setup, checkpoint download, and the AWS quote.
'''

BRIEF_PATH = Path('/content/kandinsky_aws_meeting_brief.md')
BRIEF_PATH.write_text(brief, encoding='utf-8')
print(brief)
from google.colab import files
files.download(str(BRIEF_PATH))
